# 04 - Baseline por regras e CRF

Primeira comparacao sistematica. Duas abordagens, avaliadas contra o mesmo
conjunto de teste revisado manualmente:

1. **Regras**: o proprio pre-anotador do notebook 02, sem aprendizado.
2. **CRF**: campo aleatorio condicional sobre caracteristicas de token.

A avaliacao e estrita em nivel de span: inicio, fim e tag precisam coincidir.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import collections
import random

import pandas as pd

from src.dados import (
    carregar_produtos, titulos_unicos, marcas_do_catalogo,
    salvar_jsonl, carregar_jsonl, DIR_ANOT,
)

BASE = "cellphone"

from src.preanotacao import anotar_base, anotar_titulo, _regex_marcas
from src.avaliacao import para_bio, de_bio, avaliar, erros

In [ ]:
caminho_teste = DIR_ANOT / f"{BASE}.ibyte.teste.jsonl"
assert caminho_teste.exists(), "rode o notebook 03 e faca a revisao manual antes"

teste = carregar_jsonl(caminho_teste)
treino = carregar_jsonl(DIR_ANOT / f"{BASE}.ibyte.treino_auto.jsonl")

print(f"treino (rotulo automatico): {len(treino)} titulos")
print(f"teste (revisado manualmente): {len(teste)} titulos")

## 1. Baseline: regras

Aplica o pre-anotador aos titulos do teste e compara com a referencia revisada.

In [ ]:
produtos = titulos_unicos(carregar_produtos(BASE))
marcas = _regex_marcas(marcas_do_catalogo(produtos))

predicao_regras = [
    {"text": r["text"], "entities": anotar_titulo(r["text"], marcas)}
    for r in teste
]

resultado_regras = pd.DataFrame(avaliar(teste, predicao_regras))
resultado_regras

## 2. CRF

O modelo aprende a partir do treino rotulado automaticamente. Ele nao tem acesso
aos dicionarios: precisa inferir os padroes pelas caracteristicas dos tokens.

In [ ]:
def caracteristicas(tokens, i):
    token = tokens[i]
    f = {
        "vies": 1.0,
        "minusculo": token.lower(),
        "sufixo3": token[-3:],
        "sufixo2": token[-2:],
        "prefixo2": token[:2],
        "maiuscula_inicial": token[:1].isupper(),
        "tudo_maiusculo": token.isupper(),
        "tem_digito": any(c.isdigit() for c in token),
        "so_digito": token.isdigit(),
        "alfanumerico": any(c.isdigit() for c in token) and any(c.isalpha() for c in token),
        "tamanho": len(token),
        "posicao": i,
        "primeiro": i == 0,
        "ultimo": i == len(tokens) - 1,
    }
    if i > 0:
        f.update({"-1:minusculo": tokens[i - 1].lower(),
                  "-1:tem_digito": any(c.isdigit() for c in tokens[i - 1]),
                  "-1:maiuscula": tokens[i - 1][:1].isupper()})
    if i > 1:
        f["-2:minusculo"] = tokens[i - 2].lower()
    if i < len(tokens) - 1:
        f.update({"+1:minusculo": tokens[i + 1].lower(),
                  "+1:tem_digito": any(c.isdigit() for c in tokens[i + 1])})
    if i < len(tokens) - 2:
        f["+2:minusculo"] = tokens[i + 2].lower()
    return f


def vetorizar(registros):
    X, y = [], []
    for registro in registros:
        tokens, rotulos = para_bio(registro)
        X.append([caracteristicas(tokens, i) for i in range(len(tokens))])
        y.append(rotulos)
    return X, y

In [ ]:
import sklearn_crfsuite

X_treino, y_treino = vetorizar(treino)
X_teste, y_teste = vetorizar(teste)

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=200,
    all_possible_transitions=True,
)
crf.fit(X_treino, y_treino)
print(f"tokens de treino: {sum(len(x) for x in X_treino)}")
print(f"rotulos aprendidos: {len(crf.classes_)}")

In [ ]:
previsto = crf.predict(X_teste)

predicao_crf = [
    {"text": r["text"], "entities": de_bio(r["text"], rotulos)}
    for r, rotulos in zip(teste, previsto)
]

resultado_crf = pd.DataFrame(avaliar(teste, predicao_crf))
resultado_crf

## 3. Comparacao

In [ ]:
comparacao = pd.merge(
    resultado_regras[["tag", "f1", "suporte"]].rename(columns={"f1": "regras"}),
    resultado_crf[["tag", "f1"]].rename(columns={"f1": "crf"}),
    on="tag", how="outer",
)
comparacao["diferenca"] = comparacao["crf"] - comparacao["regras"]
comparacao.sort_values("suporte", ascending=False)

In [ ]:
import matplotlib.pyplot as plt

por_tag = comparacao[~comparacao["tag"].isin(["micro", "macro"])].sort_values("suporte")
eixo = por_tag.plot.barh(x="tag", y=["regras", "crf"], figsize=(8, 6),
                         color=["#9aa5b1", "#15803d"])
eixo.set_xlabel("F1 (span estrito)")
eixo.set_ylabel("")
eixo.set_title("Desempenho por entidade")
eixo.legend(["regras", "CRF"])
plt.tight_layout()
plt.show()

## 4. Curva de aprendizado

Mostra quanto o CRF ganha com mais dados de treino. Se a curva estiver plana no
fim, anotar mais titulos traria pouco retorno; se ainda estiver subindo, o
tamanho da base e o fator limitante.

In [ ]:
tamanhos = [50, 100, 200, 400, 600, len(treino)]
curva = []

for n in tamanhos:
    if n > len(treino):
        continue
    parcial = treino[:n]
    Xp, yp = vetorizar(parcial)
    modelo = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1,
                                  max_iterations=200, all_possible_transitions=True)
    modelo.fit(Xp, yp)
    pred = [{"text": r["text"], "entities": de_bio(r["text"], rot)}
            for r, rot in zip(teste, modelo.predict(X_teste))]
    micro = [l for l in avaliar(teste, pred) if l["tag"] == "micro"][0]
    curva.append({"titulos": n, "f1": micro["f1"]})
    print(f"{n:5} titulos -> F1 {micro['f1']:.3f}")

pd.DataFrame(curva).plot(x="titulos", y="f1", marker="o", figsize=(7, 4),
                         legend=False, color="#15803d")
plt.ylabel("F1 (micro)")
plt.xlabel("titulos de treino")
plt.title("Curva de aprendizado do CRF")
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()

## 5. Analise dos erros

Os spans divergentes explicam onde cada abordagem falha.

In [ ]:
pd.DataFrame(erros(teste, predicao_crf, limite=20))

## 6. Observacoes

Preencher apos rodar:

- onde o CRF supera as regras e por que
- onde as regras ainda ganham (entidades de padrao rigido tendem a favorecer regra)
- efeito do rotulo automatico no treino
- leitura da curva de aprendizado